In [1]:
import pandas as pd

train = pd.read_csv("http://114.207.245.181:13000/txt/ratings_train.txt", sep="\t")
test = pd.read_csv("http://114.207.245.181:13000/txt/ratings_test.txt", sep="\t")

In [2]:
# na 인것은 삭제
train = train.dropna()
test = test.dropna()

train_texts = train['document'].to_list()
test_texts = test['document'].to_list()

train_labels = train['label'].to_list()
test_labels = test['label'].to_list()

train_texts[:3], train_labels[:3]

(['아 더빙.. 진짜 짜증나네요 목소리',
  '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나',
  '너무재밓었다그래서보는것을추천한다'],
 [0, 1, 0])

In [3]:
def clean_texts(texts, labels) :
    new_texts = []
    new_labels = []

    for t, l in zip(texts, labels):
        # 
        if t is not None and len(t.strip()) > 0:
            new_texts.append(t)
            new_labels.append(l)
    
    return new_texts, new_labels

train_texts, train_labels = clean_texts(train_texts, train_labels)
test_texts, test_labels = clean_texts(test_texts, test_labels)

In [4]:
# 단어 사전 생성하기
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(oov_token="<UMK>")
tokenizer.fit_on_texts(train_texts)

voca_size = len(tokenizer.word_index) + 1

In [5]:
x_train = tokenizer.texts_to_sequences(train_texts)
x_test = tokenizer.texts_to_sequences(test_texts)

In [10]:
# 숫자 값의 길이를 맞춤 pad
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

x_train_pad = pad_sequences(x_train, maxlen=100, padding="post")
x_test_pad = pad_sequences(x_test, maxlen=100, padding="post")

x_train_pad = np.array(x_train_pad)
x_test_pad = np.array(x_test_pad)

x_train_pad.shape, x_test_pad.shape

((149995, 100), (49997, 100))

In [6]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model = tf.keras.Sequential([
    Input(shape=(100,)),
    Embedding(input_dim=voca_size, output_dim=128, mask_zero=True), # mask_zero=True : 0의 값은 패딩으로 처리
    LSTM(units=64, activation="tanh", dropout=0.3, recurrent_dropout=0.3),
    Dense(units=32, activation="relu"),
    Dense(units=1, activation="sigmoid")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │    37,927,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,979,457 (144.88 MB)

 Trainable params: 37,979,457 (144.88 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
import numpy as np
y_train = np.array(train_labels)
y_test = np.array(test_labels)

y_train.shape, y_test.shape

((149995,), (49997,))

In [12]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train_pad, y_train, epochs=5, verbose=1, validation_data=(x_test_pad, y_test))

Epoch 1/5
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 492s 105ms/step - accuracy: 0.8031 - loss: 0.4060 - val_accuracy: 0.8299 - val_loss: 0.3699
Epoch 2/5
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 599s 128ms/step - accuracy: 0.9469 - loss: 0.1467 - val_accuracy: 0.8202 - val_loss: 0.4296
Epoch 3/5
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 597s 127ms/step - accuracy: 0.9799 - loss: 0.0599 - val_accuracy: 0.8138 - val_loss: 0.5023
Epoch 4/5
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 625s 133ms/step - accuracy: 0.9886 - loss: 0.0331 - val_accuracy: 0.8100 - val_loss: 0.6587
Epoch 5/5
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 657s 140ms/step - accuracy: 0.9920 - loss: 0.0224 - val_accuracy: 0.8019 - val_loss: 0.7247


In [13]:
# 예측하기
sample = "이 영화는 진짜 재밌다"

seq = tokenizer.texts_to_sequences([sample])
padded = pad_sequences(seq, maxlen=100, padding="post")

# 0 ~ 1
model.predict(padded)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step


array([[0.9865331]], dtype=float32)